In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# IMDb RESULT ANALYSIS — SINGLE BLOCK
# =========================================================
# Expected input:
# - benchmark_aggregate_results.csv
#
# Expected output format:
# official_id, query_name, n_tested_configs, n_activated_configs, DSR,
# best_config, best_group, best_design_pattern, best_p95_ms,
# top1_preserved_by_activated, activated_regret,
# best_primary_config, best_primary_p95_ms, primary_regret
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
results_csv = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/results/benchmark_aggregate_results.csv")
selected_run_phase = "hot"   # options: "hot" or "cold"
output_dir = Path("imdb_analysis_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Read aggregate results
# ---------------------------------------------------------
agg = pd.read_csv(results_csv)

required_cols = [
    "config_name",
    "activated_class",
    "benchmark_family",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "p95_latency_ms",
]
missing = [c for c in required_cols if c not in agg.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---------------------------------------------------------
# 3) Keep one run phase
# ---------------------------------------------------------
phase_df = agg[agg["run_phase"] == selected_run_phase].copy()

if phase_df.empty:
    raise ValueError(f"No rows found for run_phase={selected_run_phase!r}")

# ---------------------------------------------------------
# 4) Helper: derive official_id from IMDb query names
# ---------------------------------------------------------
def imdb_official_id(query_name: str) -> str:
    # Examples:
    # QG1_WatchItemById -> QG1
    # QG10_AdvancedSearchWatchItems -> QG10
    parts = str(query_name).split("_")
    return parts[0] if parts else str(query_name)

# ---------------------------------------------------------
# 5) Query-level summary
# ---------------------------------------------------------
rows = []

for query_name, grp in phase_df.groupby("query_name", sort=True):
    grp = grp.copy()

    grp["official_id"] = grp["query_name"].apply(imdb_official_id)

    # Best configuration across all tested configurations for this query
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]

    # Activated set:
    # Similar to your FIBEN logic, consider non-control rows as activated evidence
    activated = grp[grp["query_group"] != "control"].copy()

    # Primary-only set
    primary = grp[grp["query_group"] == "primary"].copy()

    # Counts
    n_tested_configs = grp["activated_class"].nunique()
    n_activated_configs = activated["activated_class"].nunique()

    # DSR in the same style as your previous notebook
    dsr = 1 - (n_activated_configs / n_tested_configs) if n_tested_configs > 0 else np.nan

    # Activated-family preservation / regret
    if activated.empty:
        top1_preserved_by_activated = False
        activated_regret = np.nan
    else:
        best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]
        top1_preserved_by_activated = best_activated["activated_class"] == best_all["activated_class"]
        activated_regret = (
            (best_activated["p95_latency_ms"] - best_all["p95_latency_ms"]) / best_all["p95_latency_ms"]
            if best_all["p95_latency_ms"] > 0 else np.nan
        )

    # Best primary config / primary regret
    if primary.empty:
        best_primary_config = None
        best_primary_p95_ms = np.nan
        primary_regret = np.nan
    else:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        best_primary_config = best_primary["activated_class"]
        best_primary_p95_ms = best_primary["p95_latency_ms"]
        primary_regret = (
            (best_primary_p95_ms - best_all["p95_latency_ms"]) / best_all["p95_latency_ms"]
            if best_all["p95_latency_ms"] > 0 else np.nan
        )

    rows.append({
        "official_id": imdb_official_id(query_name),
        "query_name": query_name,
        "n_tested_configs": int(n_tested_configs),
        "n_activated_configs": int(n_activated_configs),
        "DSR": float(dsr) if pd.notna(dsr) else np.nan,
        "best_config": best_all["activated_class"],
        "best_group": best_all["query_group"],
        "best_design_pattern": best_all["benchmark_family"],
        "best_p95_ms": float(best_all["p95_latency_ms"]),
        "top1_preserved_by_activated": bool(top1_preserved_by_activated),
        "activated_regret": float(activated_regret) if pd.notna(activated_regret) else np.nan,
        "best_primary_config": best_primary_config,
        "best_primary_p95_ms": float(best_primary_p95_ms) if pd.notna(best_primary_p95_ms) else np.nan,
        "primary_regret": float(primary_regret) if pd.notna(primary_regret) else np.nan,
    })

summary_df = pd.DataFrame(rows)

# ---------------------------------------------------------
# 6) Sort and display final table
# ---------------------------------------------------------
summary_df = summary_df.sort_values(by=["official_id", "query_name"]).reset_index(drop=True)

print(f"IMDb query-level summary using run_phase = {selected_run_phase!r}")
display(summary_df)

# ---------------------------------------------------------
# 7) Global summary metrics
# ---------------------------------------------------------
average_dsr = summary_df["DSR"].mean()
top1_preservation_activated = summary_df["top1_preserved_by_activated"].mean()
mean_activated_regret = summary_df["activated_regret"].dropna().mean()
mean_primary_regret = summary_df["primary_regret"].dropna().mean()

print(f"\nAverage DSR: {average_dsr}")
print(f"Top-1 preservation activated: {top1_preservation_activated}")
print(f"Mean activated regret: {mean_activated_regret}")
print(f"Mean primary regret: {mean_primary_regret}")

# ---------------------------------------------------------
# 8) Show only queries where best overall != best primary
# ---------------------------------------------------------
diff_df = summary_df[
    (
        summary_df["best_config"].fillna("None").astype(str)
        != summary_df["best_primary_config"].fillna("None").astype(str)
    )
    | summary_df["best_primary_config"].isna()
].copy()

diff_df = diff_df[
    [
        "official_id",
        "query_name",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
    ]
].reset_index(drop=True)

print()
display(diff_df)

# ---------------------------------------------------------
# 9) Save outputs
# ---------------------------------------------------------
summary_df.to_csv(output_dir / f"imdb_summary_{selected_run_phase}.csv", index=False)
diff_df.to_csv(output_dir / f"imdb_diff_best_vs_primary_{selected_run_phase}.csv", index=False)

print(f"\nSaved outputs in: {output_dir.resolve()}")


IMDb query-level summary using run_phase = 'hot'


,official_id,query_name,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
0,QG1,QG1_WatchItemById,9,6,0.333333,G9,control,containment_family,0.213258,False,0.004347,G0,0.271723,0.274153
1,QG10,QG10_AdvancedSearchWatchItems,9,9,0.000000,G9,secondary_affected,containment_family,299.802023,True,0.000000,G3,300.348337,0.001822
2,QG2,QG2_WatchItemByTitle,9,6,0.333333,G8,control,containment_family,0.309479,False,0.022872,G0,0.316557,0.022872
3,QG3,QG3_RecommendationByGenreAndSubtype,9,9,0.000000,G8,secondary_affected,containment_family,10.417329,True,0.000000,G0,10.727311,0.029756
4,QG4,QG4_AllPersonsOfTypeForWatchItem,9,6,0.333333,G6,primary,associative_family,0.202353,True,0.000000,G6,0.202353,0.000000
5,QG5,QG5_AllPersonsForEpisodesOfSeries,9,9,0.000000,G4,primary,associative_family,367.568908,True,0.000000,G4,367.568908,0.000000
6,QG6,QG6_EpisodesOfSeries,9,4,0.555556,G7,primary,containment_family,2.271900,True,0.000000,G7,2.271900,0.000000
7,QG7,QG7_UpdateWatchItemMetadata,9,6,0.333333,G8,control,containment_family,0.234738,False,0.008179,G0,0.304322,0.296431
8,QG8,QG8_AddPersonRoleToWatchItem,9,6,0.333333,G3,secondary_affected,root_local_family,0.373201,True,0.000000,G0,0.375635,0.006522
9,QG9,QG9_TopRatedSeriesByGenre,9,9,0.000000,G7,secondary_affected,containment_family,4.078044,True,0.000000,G2,159.413392,38.090653



Average DSR: 0.22222222222222224
Top-1 preservation activated: 0.7
Mean activated regret: 0.0035398418557877854
Mean primary regret: 3.872221049570096



,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
0,QG1,QG1_WatchItemById,G9,containment_family,0.213258,G0,0.271723,0.274153
1,QG10,QG10_AdvancedSearchWatchItems,G9,containment_family,299.802023,G3,300.348337,0.001822
2,QG2,QG2_WatchItemByTitle,G8,containment_family,0.309479,G0,0.316557,0.022872
3,QG3,QG3_RecommendationByGenreAndSubtype,G8,containment_family,10.417329,G0,10.727311,0.029756
4,QG7,QG7_UpdateWatchItemMetadata,G8,containment_family,0.234738,G0,0.304322,0.296431
5,QG8,QG8_AddPersonRoleToWatchItem,G3,root_local_family,0.373201,G0,0.375635,0.006522
6,QG9,QG9_TopRatedSeriesByGenre,G7,containment_family,4.078044,G2,159.413392,38.090653



Saved outputs in: /home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/imdb_analysis_outputs
